In [ ]:
# AutoDIAL Full-Mamba HADA (SSM-DAN) — OpenBMI chronological LOSO, one target block.
# Enable GPU + Internet. Attach the OpenBMI cache dataset (openbmi_S01..S54.npz,
# from kaggle_prepare_openbmi_cache) via Add Input; without it the subjects are
# downloaded (~61 GB) and preprocessed first, which takes hours.
# Every fold trains on all 53 other subjects; this notebook runs the target
# subjects listed in TARGET_SUBJECTS.
import codecs
import os
import re
import select
import shutil
import subprocess
import sys
import time
from pathlib import Path

# ---------------- Job configuration ----------------
# 'autodial' = learned source/target statistic mixing (main method);
# 'none'     = original HADA without AutoDIAL, for a same-branch baseline.
FEATURE_ALIGNMENT = 'autodial'
TARGET_SUBJECTS = list(range(1, 19))
JOB_TAG = 's01_s18'
# Stop starting new folds when the next one would not finish in time
# (Kaggle sessions end at 12 h; setup and archiving need the rest).
MAX_RUN_HOURS = 11.0

DATASET = 'openbmi'
BRANCH = 'feature/hada-autodial-openbmi'
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
RUN_TAG = f"{DATASET}_{FEATURE_ALIGNMENT}_{JOB_TAG}"
REPO_PATH = Path(f"/kaggle/working/tcformer-autodial-{DATASET}-{JOB_TAG}")
# Raw downloads (only if no cache is attached) stay outside the saved output.
MNE_DATA = Path("/tmp/mne_data")
CACHE_DIR = Path("/kaggle/working/openbmi_cache")
TRAIN_LOG = Path(f"/kaggle/working/autodial_{RUN_TAG}.log")
RESULT_ARCHIVE = Path(f"/kaggle/working/autodial_{RUN_TAG}_results")


def run(command, cwd=None, env=None, stream=False, log_path=None):
    command = list(map(str, command))
    print("+", " ".join(command), flush=True)
    if not stream:
        subprocess.run(
            command,
            cwd=str(cwd) if cwd else None,
            env=env,
            check=True,
        )
        return

    process = subprocess.Popen(
        command,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    started_at = time.monotonic()
    with open(log_path, "w", encoding="utf-8") as log_file:
        while True:
            ready, _, _ = select.select([process.stdout], [], [], 30)
            if ready:
                chunk = os.read(process.stdout.fileno(), 4096)
                if not chunk:
                    break
                output = decoder.decode(chunk)
                print(output, end="", flush=True)
                log_file.write(output)
                log_file.flush()
            elif process.poll() is None:
                elapsed = (time.monotonic() - started_at) / 60
                heartbeat = f"[Kaggle heartbeat] running | elapsed={elapsed:.1f}m\n"
                print(heartbeat, end="", flush=True)
                log_file.write(heartbeat)
                log_file.flush()
            else:
                break

        remaining = decoder.decode(b"", final=True)
        if remaining:
            print(remaining, end="", flush=True)
            log_file.write(remaining)

    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(
            f"Training exited with code {return_code}. Full log: {log_path}"
        )


# ---------------- Clone branch and verify the implementation ----------------
run([sys.executable, "-m", "pip", "install", "-q", "uv"])
if (REPO_PATH / ".git").is_dir():
    run(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO_PATH)
    run(["git", "fetch", "origin", BRANCH], cwd=REPO_PATH)
    # A previous run edits the YAML locally; reset so reruns are safe.
    run(["git", "checkout", "--force", BRANCH], cwd=REPO_PATH)
    run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_PATH)
else:
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_PATH])
run(["git", "log", "-1", "--oneline"], cwd=REPO_PATH)

pipeline_text = (REPO_PATH / "train_pipeline.py").read_text(encoding="utf-8")
model_text = (REPO_PATH / "models/hada_tcformer.py").read_text(encoding="utf-8")
datamodule_text = (REPO_PATH / "utils/get_datamodule_cls.py").read_text(encoding="utf-8")
assert "ModelCheckpoint" in pipeline_text and 'monitor="val_acc"' in pipeline_text
assert "class AutoDIALFeatureNorm" in model_text
assert "openbmi_loso" in datamodule_text
assert "def score_without_tta" in pipeline_text and "MAX_RUN_HOURS" in pipeline_text
openbmi_text = (REPO_PATH / "datamodules/openbmi.py").read_text(encoding="utf-8")
assert "source_val(session 2)" in openbmi_text
print("Verified: AutoDIAL model, OpenBMI chronological LOSO (source S1 train / S2 val; target S1 unlabeled / S2 test), best source-validation checkpoint.")

UV = shutil.which("uv") or "uv"
run([UV, "venv", "--clear", "--python", "3.10", ".venv"], cwd=REPO_PATH)
PYTHON = REPO_PATH / ".venv/bin/python"
run([UV, "pip", "install", "--python", PYTHON, "torch==2.7.1", "torchvision==0.22.1",
     "--index-url", "https://download.pytorch.org/whl/cu126"])
run([UV, "pip", "install", "--python", PYTHON, "-r", "requirements.txt"], cwd=REPO_PATH)

# ---------------- Config override (subjects, alignment mode, compact logs) -------
override = (
    "import yaml; from pathlib import Path; "
    "p=Path('configs/hada_tcformer.yaml'); c=yaml.safe_load(p.read_text()); "
    f"c['subject_ids']={TARGET_SUBJECTS!r}; "
    "c['log_every_n_epochs']=10; "
    "c['enable_progress_bar']=False; "
    "c['model_kwargs']['log_every_n_batches']=0; "
    f"c['model_kwargs']['feature_alignment']={FEATURE_ALIGNMENT!r}; "
    "p.write_text(yaml.safe_dump(c, sort_keys=False))"
)
run([PYTHON, "-c", override], cwd=REPO_PATH)
run(["nvidia-smi"])
run([PYTHON, "-c", "import torch; print('PyTorch:', torch.__version__); "
     "print('CUDA:', torch.cuda.is_available()); "
     "assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'; "
     "print('GPU:', torch.cuda.get_device_name(0))"])

# ---------------- Train / test ----------------
MNE_DATA.mkdir(parents=True, exist_ok=True)
results_dir = REPO_PATH / "results"
if results_dir.exists():
    # A rerun in the same session must not mix old and new result folders.
    shutil.rmtree(results_dir)
environment = os.environ.copy()
environment.update({
    "PYTHONUNBUFFERED": "1",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "MPLBACKEND": "Agg",
    "MNE_DATA": str(MNE_DATA),
    "MNE_DATASETS_BNCI_PATH": str(MNE_DATA),
    "MNE_DATASETS_LEE2019_MI_PATH": str(MNE_DATA),
    "OPENBMI_CACHE_DIR": str(CACHE_DIR),
    "OPENBMI_CACHE_INPUT": "/kaggle/input",
    "MAX_RUN_HOURS": str(MAX_RUN_HOURS),
})
cached = sorted(Path("/kaggle/input").rglob("openbmi_S*.npz")) if Path("/kaggle/input").is_dir() else []
print(f"Attached OpenBMI cache files: {len(cached)}/54", flush=True)
if len(cached) < 54:
    print("WARNING: missing subjects will be downloaded and preprocessed first.", flush=True)
print(f"===== AUTODIAL FULL-MAMBA HADA | OpenBMI | 54 SUBJECTS | 2 CLASSES | 62 CH | 100 Hz | alignment={FEATURE_ALIGNMENT} "
      f"| TARGETS={TARGET_SUBJECTS} =====", flush=True)
print("Protocol: all 53 other subjects are sources: session 1 trains, session 2 validates (best checkpoint). Target: session 1 is unlabeled UDA data; session 2 is IM-TTA + AutoDIAL recalibration + primary test. The same model is also scored without IM-TTA.", flush=True)
print("Live log:", TRAIN_LOG, flush=True)
run([PYTHON, "-u", "train_pipeline.py", "--model", "hada_tcformer", "--dataset", DATASET,
     "--loso", "--gpu_id", "0"], cwd=REPO_PATH, env=environment, stream=True,
    log_path=TRAIN_LOG)

# ---------------- Summary per completed target ----------------
log_text = TRAIN_LOG.read_text(encoding="utf-8", errors="replace")
alpha_prefix = "AutoDIAL alpha per feature group: "
current, alpha_by_subject, completed = None, {}, []
for line in log_text.splitlines():
    if line.startswith(">>> Training on subject: "):
        current = int(line.rsplit(":", 1)[1])
    elif alpha_prefix in line and current is not None:
        alpha_by_subject[current] = line.split(alpha_prefix, 1)[1].strip()   # last = after IM-TTA
    match = re.match(r">>> Fold of target (\d+) took", line)
    if match:
        completed.append(int(match.group(1)))
for subject_id in completed:
    print(f"S{subject_id:02d} final alpha (CNN groups 1-3, SSM group): {alpha_by_subject.get(subject_id)}")

archive = shutil.make_archive(str(RESULT_ARCHIVE), "zip", root_dir=REPO_PATH, base_dir="results")
remaining = [s for s in TARGET_SUBJECTS if s not in completed]
print("Completed targets:", completed)
print("Remaining targets (rerun with TARGET_SUBJECTS = this list):", remaining)
print("Kaggle output archive:", archive)
print("Kaggle output log:", TRAIN_LOG)
